# Qwen2.5-3B CPT-LoRA Grounding Eval

Audit notebook for the merged CPT checkpoint produced from the SID-v2 Qwen2.5-3B LoRA CPT run.

Checks:

- SID -> title/year/genres exact recovery;
- SID -> short plot recovery;
- title/year -> short plot recovery;
- CSV exports for manual review.

The plot checks are heuristic: descriptions may be valid paraphrases, so the notebook reports lexical overlap/F1 and keeps side-by-side text samples.

## 1. Imports and config

Change `META_EVAL_N` / `DESC_EVAL_N` to `None` for all items.

In [1]:
from pathlib import Path
from importlib import metadata
from packaging.version import Version
import json
import math
import os
import random
import re
import sys

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

RUN_NAME = "cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1"
ARTIFACT_DIR = ROOT / "data/processed/artifacts" / RUN_NAME
MODEL_DIR = ARTIFACT_DIR / "final_merged"
ITEM_PROFILE_PATH = ROOT / "data/processed/item_features/qwen4b_audited_v1_item_profiles.parquet"
SID_ARRAY_PATH = ROOT / "runs/qwen4b_rqvae_sid_v2_plum/SIDs_best.npy"
OUT_DIR = ROOT / "reports/cpt_grounding/qwen2_5_3b_sid_v2"

SEED = 42
META_EVAL_N = 300      # set None for all 3706 movies
DESC_EVAL_N = 120      # set None for all movies with descriptions
GEN_BATCH_SIZE = 8
MAX_META_NEW_TOKENS = 80
MAX_DESC_NEW_TOKENS = 96

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == "cuda" else torch.float32)

OUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)

print("root:", ROOT)
print("model:", MODEL_DIR)
print("device:", DEVICE, DTYPE)
print("transformers:", metadata.version("transformers"))

root: C:\Users\User\plum-ml1m-repro
model: C:\Users\User\plum-ml1m-repro\data\processed\artifacts\cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1\final_merged
device: cuda torch.bfloat16
transformers: 4.57.6


## 2. Load item universe

In [2]:
profiles = pd.read_parquet(ITEM_PROFILE_PATH).sort_values("item_idx").reset_index(drop=True)
sids = np.load(SID_ARRAY_PATH)

assert sids.shape == (len(profiles), 4)
profiles["title_clean"] = profiles["title_clean"].fillna(profiles["title"].astype(str))
profiles["year_text"] = profiles["year_text"].fillna(profiles["year"].astype(str))
profiles["genres_text"] = profiles["genres_text"].fillna(profiles["genres"].astype(str).str.replace("|", ", ", regex=False))
profiles["description_text"] = profiles["description_text"].fillna("")

profiles[["item_idx", "movie_id", "title", "title_clean", "year_text", "genres_text", "description_text"]].head()

,item_idx,movie_id,title,title_clean,year_text,genres_text,description_text
0,0,1,Toy Story (1995),Toy Story,1995,"Animation, Children's, Comedy","Plot overview: Led by Woody, Andy's toys live ..."
1,1,2,Jumanji (1995),Jumanji,1995,"Adventure, Children's, Fantasy",Plot overview: When siblings Judy and Peter di...
2,2,3,Grumpier Old Men (1995),Grumpier Old Men,1995,"Comedy, Romance",Plot overview: A family wedding reignites the ...
3,3,4,Waiting to Exhale (1995),Waiting to Exhale,1995,"Comedy, Drama","Plot overview: Cheated on, mistreated and step..."
4,4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995,Comedy,Plot overview: Just when George Banks has reco...


## 3. Load CPT model

In [3]:
def load_tokenizer(path):
    try:
        return AutoTokenizer.from_pretrained(path, use_fast=True, fix_mistral_regex=True, local_files_only=True)
    except TypeError:
        return AutoTokenizer.from_pretrained(path, use_fast=True, local_files_only=True)


def load_causal_lm(path):
    kwargs = {"dtype": DTYPE, "local_files_only": True, "trust_remote_code": False}
    try:
        return AutoModelForCausalLM.from_pretrained(path, **kwargs)
    except TypeError:
        kwargs["torch_dtype"] = kwargs.pop("dtype")
        return AutoModelForCausalLM.from_pretrained(path, **kwargs)


tokenizer = load_tokenizer(MODEL_DIR)
tokenizer.padding_side = "left"
model = load_causal_lm(MODEL_DIR).to(DEVICE)
model.eval()

print("vocab:", len(tokenizer))
print("pad/eos/bos:", tokenizer.pad_token_id, tokenizer.eos_token_id, tokenizer.bos_token_id)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

vocab: 152049
pad/eos/bos: 151667 151666 151665


## 4. Prompt helpers

In [4]:
BOS = "<bos>"
EOS = "<eos>"
META_OPEN = "<meta>"
DESC_OPEN = "<desc>"
TASK_META = "<task_meta>"
TASK_DESC = "<task_desc>"


def sid_tokens(sid):
    return [f"<sid_{level}_{int(code)}>" for level, code in enumerate(sid)]


def sid_text(item_idx):
    return " ".join(sid_tokens(sids[int(item_idx)]))


def meta_prompt(item_idx):
    return f"{BOS} {TASK_META} {META_OPEN} Movie {sid_text(item_idx)} has title:"


def sid_desc_prompt(row):
    return f"{BOS} {TASK_DESC} {DESC_OPEN} Movie {sid_text(int(row.item_idx))}. Short plot:"


def sid_title_desc_prompt(row):
    return f"{BOS} {TASK_DESC} {DESC_OPEN} Movie {sid_text(int(row.item_idx))}. Title: {row.title_clean}. Short plot:"


def title_desc_prompt(row):
    return f"{BOS} {TASK_DESC} {DESC_OPEN} Title: {row.title_clean}. Release year: {row.year_text}. Short plot:"


def batch_generate(prompts, max_new_tokens, batch_size=GEN_BATCH_SIZE):
    outputs = []
    for start in tqdm(range(0, len(prompts), batch_size), desc="generate", leave=False):
        batch = prompts[start:start + batch_size]
        enc = tokenizer(batch, return_tensors="pt", padding=True).to(DEVICE)
        with torch.inference_mode():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                repetition_penalty=1.03,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        outputs.extend(tokenizer.batch_decode(out, skip_special_tokens=False))
    return outputs

## 5. Parsing and metrics

In [5]:
STOPWORDS = set("""
a an and are as at be by for from has have he her his in into is it its of on or she that the their them they this to was were when where who with without young old after before while during through over under about into out up down across only just new one two three film movie story plot overview
""".split())


def clean_text(x):
    return re.sub(r"\s+", " ", str(x)).strip()


def norm_scalar(x):
    x = clean_text(x).lower()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    return re.sub(r"\s+", " ", x).strip()


def genre_set(x):
    return {norm_scalar(g) for g in str(x).replace("|", ",").split(",") if norm_scalar(g)}


def content_tokens(x):
    toks = re.findall(r"[a-z0-9']+", str(x).lower())
    return [t for t in toks if len(t) >= 3 and t not in STOPWORDS]


def token_f1(pred, ref):
    pred_tokens = content_tokens(pred)
    ref_tokens = content_tokens(ref)
    if not pred_tokens or not ref_tokens:
        return 0.0
    pred_counts = pd.Series(pred_tokens).value_counts().to_dict()
    ref_counts = pd.Series(ref_tokens).value_counts().to_dict()
    overlap = sum(min(pred_counts.get(t, 0), ref_counts.get(t, 0)) for t in set(pred_counts) | set(ref_counts))
    precision = overlap / max(len(pred_tokens), 1)
    recall = overlap / max(len(ref_tokens), 1)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)


def extract_between(text, start_patterns, end_patterns):
    s = str(text)
    lower = s.lower()
    start_pos = -1
    start_len = 0
    for pat in start_patterns:
        pos = lower.find(pat.lower())
        if pos >= 0:
            start_pos = pos
            start_len = len(pat)
            break
    if start_pos < 0:
        return ""
    tail = s[start_pos + start_len:]
    tail_lower = tail.lower()
    end_pos = len(tail)
    for pat in end_patterns:
        pos = tail_lower.find(pat.lower())
        if pos >= 0:
            end_pos = min(end_pos, pos)
    return clean_text(tail[:end_pos].strip(" .:\n\t"))


def parse_metadata(output):
    title = extract_between(output, ["has title:"], ["Release year:", "Year:", "Genres:", "</meta>", "<eos>"])
    year = extract_between(output, ["Release year:", "Year:"], ["Genres:", "</meta>", "<eos>"])
    genres = extract_between(output, ["Genres:", "Genre labels:"], ["</meta>", "<eos>"])
    year_match = re.search(r"\d{4}", year)
    return {
        "pred_title": clean_text(title),
        "pred_year": year_match.group(0) if year_match else clean_text(year),
        "pred_genres": clean_text(genres),
    }


def extract_description(output):
    desc = extract_between(output, ["Short plot:", "Plot overview:", "Plot:"], ["</desc>", "</meta>", "<eos>"])
    desc = re.sub(r"^(plot\s+overview|overview):\s*", "", desc, flags=re.I).strip()
    return desc

## 6. SID -> title/year/genres exact recovery

This is the strictest grounding test. It checks whether the model can decode the metadata attached to a SID.

In [6]:
def sample_items(n, only_with_description=False):
    df = profiles.copy()
    if only_with_description:
        mask = df["description_text"].astype(str).str.strip().ne("")
        df = df[mask].copy()
    if n is None or n >= len(df):
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=SEED).sort_values("item_idx").reset_index(drop=True)


meta_items = sample_items(META_EVAL_N)
meta_outputs = batch_generate([meta_prompt(int(r.item_idx)) for r in meta_items.itertuples(index=False)], MAX_META_NEW_TOKENS)

meta_records = []
for row, output in zip(meta_items.itertuples(index=False), meta_outputs):
    parsed = parse_metadata(output)
    title_ok = norm_scalar(parsed["pred_title"]) == norm_scalar(row.title_clean)
    year_ok = str(parsed["pred_year"]).strip() == str(row.year_text).strip()
    genres_ok = genre_set(parsed["pred_genres"]) == genre_set(row.genres_text)
    meta_records.append({
        "item_idx": int(row.item_idx),
        "movie_id": int(row.movie_id),
        "sid": sid_text(int(row.item_idx)),
        "true_title": row.title_clean,
        "pred_title": parsed["pred_title"],
        "title_ok": bool(title_ok),
        "true_year": row.year_text,
        "pred_year": parsed["pred_year"],
        "year_ok": bool(year_ok),
        "true_genres": row.genres_text,
        "pred_genres": parsed["pred_genres"],
        "genres_ok": bool(genres_ok),
        "all_ok": bool(title_ok and year_ok and genres_ok),
        "output": output,
    })

meta_eval = pd.DataFrame(meta_records)
meta_eval.to_csv(OUT_DIR / "sid_to_metadata_eval.csv", index=False, encoding="utf-8-sig")

meta_summary = {
    "n": int(len(meta_eval)),
    "title_accuracy": float(meta_eval["title_ok"].mean()),
    "year_accuracy": float(meta_eval["year_ok"].mean()),
    "genres_accuracy": float(meta_eval["genres_ok"].mean()),
    "all_exact_accuracy": float(meta_eval["all_ok"].mean()),
    "errors": int((~meta_eval["all_ok"]).sum()),
}
print(json.dumps(meta_summary, indent=2))
meta_eval[~meta_eval["all_ok"]].head(20)

generate:   0%|          | 0/38 [00:00<?, ?it/s]

{
  "n": 300,
  "title_accuracy": 0.7166666666666667,
  "year_accuracy": 0.7366666666666667,
  "genres_accuracy": 0.8166666666666667,
  "all_exact_accuracy": 0.71,
  "errors": 87
}


,item_idx,movie_id,sid,true_title,pred_title,title_ok,true_year,pred_year,year_ok,true_genres,pred_genres,genres_ok,all_ok,output
2,17,18,<sid_0_453> <sid_1_223> <sid_2_33> <sid_3_62>,Four Rooms,"Hollywood Knights, The",False,1995,1980,False,Thriller,Comedy,False,False,<bos> <task_meta> <meta> Movie <sid_0_453> <si...
9,93,96,<sid_0_461> <sid_1_21> <sid_2_10> <sid_3_61>,In the Bleak Midwinter,"Englishman Who Went Up a Hill, But Came Down a...",False,1995,1995,True,Comedy,"Comedy, Romance",False,False,<bos> <task_meta> <meta> Movie <sid_0_461> <si...
10,109,113,<sid_0_453> <sid_1_53> <sid_2_6> <sid_3_5>,Before and After,Leaving Las Vegas,False,1996,1995,False,"Drama, Mystery","Drama, Romance",False,False,<bos> <task_meta> <meta> Movie <sid_0_453> <si...
15,149,155,<sid_0_445> <sid_1_139> <sid_2_117> <sid_3_2>,Beyond Rangoon,Blood & Wine,False,1995,1997,False,"Drama, War",Drama,False,False,<bos> <task_meta> <meta> Movie <sid_0_445> <si...
21,196,202,<sid_0_78> <sid_1_25> <sid_2_28> <sid_3_4>,Total Eclipse,I Don't Want to Talk About It (De eso no se ha...,False,1995,1993,False,"Drama, Romance",Drama,False,False,<bos> <task_meta> <meta> Movie <sid_0_78> <sid...
31,298,307,<sid_0_78> <sid_1_15> <sid_2_86> <sid_3_36>,Three Colors: Blue,Nights of Cabiria (Le Notti di Cabiria),False,1993,1957,False,Drama,Drama,True,False,<bos> <task_meta> <meta> Movie <sid_0_78> <sid...
33,315,325,<sid_0_284> <sid_1_200> <sid_2_78> <sid_3_14>,National Lampoon's Senior Trip,National Lampoon's Last Resort,False,1995,1994,False,Comedy,Comedy,True,False,<bos> <task_meta> <meta> Movie <sid_0_284> <si...
35,325,335,<sid_0_453> <sid_1_220> <sid_2_94> <sid_3_30>,"Underneath, The","Underneath, The",True,1995,1995,True,"Mystery, Thriller",Thriller,False,False,<bos> <task_meta> <meta> Movie <sid_0_453> <si...
37,358,368,<sid_0_109> <sid_1_171> <sid_2_34> <sid_3_41>,Maverick,Lonely Are the Brave,False,1994,1962,False,"Action, Comedy, Western","Drama, Western",False,False,<bos> <task_meta> <meta> Movie <sid_0_109> <si...
41,393,407,<sid_0_154> <sid_1_216> <sid_2_26> <sid_3_43>,In the Mouth of Madness,I Know What You Did Last Summer,False,1995,1997,False,"Horror, Thriller","Horror, Mystery, Thriller",False,False,<bos> <task_meta> <meta> Movie <sid_0_154> <si...


## 7. Description recovery by SID and by title/year

Exact description matching is too strict, so this cell reports lexical F1 against the audited overview and exports examples for manual review.

In [7]:
desc_items = sample_items(DESC_EVAL_N, only_with_description=True)

sid_desc_outputs = batch_generate([sid_desc_prompt(r) for r in desc_items.itertuples(index=False)], MAX_DESC_NEW_TOKENS)
sid_title_desc_outputs = batch_generate([sid_title_desc_prompt(r) for r in desc_items.itertuples(index=False)], MAX_DESC_NEW_TOKENS)
title_desc_outputs = batch_generate([title_desc_prompt(r) for r in desc_items.itertuples(index=False)], MAX_DESC_NEW_TOKENS)

desc_records = []
for row, sid_output, sid_title_output, title_output in zip(desc_items.itertuples(index=False), sid_desc_outputs, sid_title_desc_outputs, title_desc_outputs):
    ref = clean_text(str(row.description_text).replace("Plot overview:", "").strip())
    sid_pred = extract_description(sid_output)
    sid_title_pred = extract_description(sid_title_output)
    title_pred = extract_description(title_output)
    desc_records.append({
        "item_idx": int(row.item_idx),
        "movie_id": int(row.movie_id),
        "sid": sid_text(int(row.item_idx)),
        "title": row.title_clean,
        "year": row.year_text,
        "genres": row.genres_text,
        "reference_description": ref,
        "sid_only_description": sid_pred,
        "sid_title_description": sid_title_pred,
        "title_year_description": title_pred,
        "sid_only_f1": token_f1(sid_pred, ref),
        "sid_title_f1": token_f1(sid_title_pred, ref),
        "title_year_f1": token_f1(title_pred, ref),
        "sid_output": sid_output,
        "sid_title_output": sid_title_output,
        "title_output": title_output,
    })

desc_eval = pd.DataFrame(desc_records)
desc_eval.to_csv(OUT_DIR / "description_recovery_eval.csv", index=False, encoding="utf-8-sig")

desc_summary = {
    "n": int(len(desc_eval)),
    "sid_only_f1_mean": float(desc_eval["sid_only_f1"].mean()),
    "sid_only_f1_median": float(desc_eval["sid_only_f1"].median()),
    "sid_title_f1_mean": float(desc_eval["sid_title_f1"].mean()),
    "sid_title_f1_median": float(desc_eval["sid_title_f1"].median()),
    "title_year_f1_mean": float(desc_eval["title_year_f1"].mean()),
    "title_year_f1_median": float(desc_eval["title_year_f1"].median()),
    "sid_title_beats_sid_only_rate": float((desc_eval["sid_title_f1"] > desc_eval["sid_only_f1"]).mean()),
    "title_year_beats_sid_only_rate": float((desc_eval["title_year_f1"] > desc_eval["sid_only_f1"]).mean()),
}
print(json.dumps(desc_summary, indent=2))
desc_eval.sort_values("sid_title_f1", ascending=False).head(10)[[
    "title", "year", "sid_only_f1", "sid_title_f1", "title_year_f1", "reference_description", "sid_title_description", "title_year_description"
]]

generate:   0%|          | 0/15 [00:00<?, ?it/s]

generate:   0%|          | 0/15 [00:00<?, ?it/s]

generate:   0%|          | 0/15 [00:00<?, ?it/s]

{
  "n": 120,
  "sid_only_f1_mean": 0.03153446230310105,
  "sid_only_f1_median": 0.0,
  "sid_title_f1_mean": 0.08484398788183928,
  "sid_title_f1_median": 0.059714795008912656,
  "title_year_f1_mean": 0.08792363439239673,
  "title_year_f1_median": 0.06612021857923497,
  "sid_title_beats_sid_only_rate": 0.6583333333333333,
  "title_year_beats_sid_only_rate": 0.6666666666666666
}


,title,year,sid_only_f1,sid_title_f1,title_year_f1,reference_description,sid_title_description,title_year_description
37,Groundhog Day,1993,0.055556,0.439024,0.439024,"A narcissistic TV weatherman, along with his a...",A cynical TV weatherman is sent to Groundhog D...,A cynical TV weatherman is sent to Groundhog D...
59,In the Heat of the Night,1967,0.000000,0.347826,0.476190,An African American detective is asked to inve...,A small-town police chief is sent to investiga...,A black detective is called to investigate a m...
47,Tetsuo II: Body Hammer,1992,0.000000,0.333333,0.166667,Tetsuo II: Body Hammer is a 1992 Japanese scie...,Tetsuo II: Body Hammer is a Japanese science f...,"Tetsuo II is a sequel to Tetsuo I: Iron Man, a..."
55,"Land Girls, The",1998,0.115385,0.272727,0.279070,"During World War II, the organisation ""The Wom...",A group of women are sent to the countryside i...,A group of women are sent to a farm in the Eng...
35,Amadeus,1984,0.000000,0.260870,0.260870,The incredible story of genius musician Wolfga...,"The story of Wolfgang Amadeus Mozart, from his...","The story of Wolfgang Amadeus Mozart, from his..."
36,"Deer Hunter, The",1978,0.113208,0.244898,0.244898,A group of working-class friends decides to en...,A group of young men from a working-class Penn...,A group of young men from a working-class Penn...
90,Thumbelina,1994,0.000000,0.240000,0.160000,The tiny girl meets a fairy prince who saves h...,A tiny girl with a flower for a nose and a but...,A tiny girl with a flower for a nose and a but...
87,Othello,1952,0.054054,0.222222,0.227273,"Desdemona, daughter of a Venetian aristocrat, ...",Othello is a Moorish general who has been prom...,Othello is a Moorish general who is promoted t...
68,Secret Agent,1936,0.095238,0.206897,0.266667,After three British agents are assigned to ass...,A British agent is sent to Russia to assassina...,A British spy is sent to Russia to assassinate...
105,Thelma & Louise,1991,0.035714,0.204082,0.160000,"Whilst on a short weekend getaway, Louise shoo...","Two women, Thelma and Louise, are on the run f...","Two women, Thelma and Louise, are on the run f..."


## 8. Worst cases for manual audit

In [8]:
print("metadata errors:", int((~meta_eval["all_ok"]).sum()))
display(meta_eval[~meta_eval["all_ok"]][[
    "item_idx", "title_ok", "year_ok", "genres_ok", "true_title", "pred_title", "true_year", "pred_year", "true_genres", "pred_genres"
]].head(30))

print("lowest SID-only description F1")
display(desc_eval.sort_values("sid_only_f1").head(15)[[
    "title", "year", "genres", "sid_only_f1", "reference_description", "sid_only_description"
]])

print("lowest SID+title description F1")
display(desc_eval.sort_values("sid_title_f1").head(15)[[
    "title", "year", "genres", "sid_title_f1", "reference_description", "sid_title_description"
]])

print("lowest title/year description F1")
display(desc_eval.sort_values("title_year_f1").head(15)[[
    "title", "year", "genres", "title_year_f1", "reference_description", "title_year_description"
]])

metadata errors: 87


,item_idx,title_ok,year_ok,genres_ok,true_title,pred_title,true_year,pred_year,true_genres,pred_genres
2,17,False,False,False,Four Rooms,"Hollywood Knights, The",1995,1980,Thriller,Comedy
9,93,False,True,False,In the Bleak Midwinter,"Englishman Who Went Up a Hill, But Came Down a...",1995,1995,Comedy,"Comedy, Romance"
10,109,False,False,False,Before and After,Leaving Las Vegas,1996,1995,"Drama, Mystery","Drama, Romance"
15,149,False,False,False,Beyond Rangoon,Blood & Wine,1995,1997,"Drama, War",Drama
21,196,False,False,False,Total Eclipse,I Don't Want to Talk About It (De eso no se ha...,1995,1993,"Drama, Romance",Drama
31,298,False,False,True,Three Colors: Blue,Nights of Cabiria (Le Notti di Cabiria),1993,1957,Drama,Drama
33,315,False,False,True,National Lampoon's Senior Trip,National Lampoon's Last Resort,1995,1994,Comedy,Comedy
35,325,True,True,False,"Underneath, The","Underneath, The",1995,1995,"Mystery, Thriller",Thriller
37,358,False,False,False,Maverick,Lonely Are the Brave,1994,1962,"Action, Comedy, Western","Drama, Western"
41,393,False,False,False,In the Mouth of Madness,I Know What You Did Last Summer,1995,1997,"Horror, Thriller","Horror, Mystery, Thriller"


lowest SID-only description F1


,title,year,genres,sid_only_f1,reference_description,sid_only_description
0,Cutthroat Island,1995,"Action, Adventure, Romance",0.0,"Morgan Adams and her slave, William Shaw, are ...",A group of young people from a small town in t...
72,Cocoon: The Return,1988,"Comedy, Sci-Fi",0.0,The old age pensioners that left at the end of...,"A group of friends, including a gay man, a les..."
71,Rushmore,1998,Comedy,0.0,When a beautiful first-grade teacher arrives a...,"A young man, who has just been fired from his ..."
67,Weird Science,1985,Comedy,0.0,"Two unpopular teenagers, Gary and Wyatt, fail ...",A group of high school students from a small t...
66,"Dead Zone, The",1983,"Horror, Thriller",0.0,Johnny Smith is a schoolteacher with his whole...,"A group of friends, including a young woman wh..."
65,Nineteen Eighty-Four,1984,"Drama, Sci-Fi",0.0,George Orwell's novel of a totalitarian future...,A group of scientists and their families are t...
63,"Negotiator, The",1998,"Action, Thriller",0.0,The police try to arrest expert hostage negoti...,A group of young men from a small town in the ...
62,Seven Samurai (The Magnificent Seven) (Shichin...,1954,"Action, Drama",0.0,A samurai answers a village's request for prot...,"A young Jewish boy, Max, is sent to live with ..."
73,Nothing in Common,1986,Comedy,0.0,A successful Chicago advertising executive see...,"A young woman, who has just moved to New York ..."
60,Klute,1971,"Drama, Mystery",0.0,This acclaimed thriller stars Jane Fonda as Br...,A young woman is kidnapped by a serial killer ...


lowest SID+title description F1


,title,year,genres,sid_title_f1,reference_description,sid_title_description
52,"Newton Boys, The",1998,"Crime, Drama",0.0,Four Newton brothers are a poor farmer family ...,A young man is sent to a juvenile detention ce...
106,...And Justice for All,1979,"Drama, Thriller",0.0,An ethical Baltimore defense lawyer disgusted ...,A man is falsely accused of a crime and must f...
110,King Creole,1958,"Drama, Musical",0.0,"Danny Fisher, young delinquent, flunks out of ...","A young man, who is a descendant of the origin..."
29,Macao,1952,Adventure,0.0,"On the ferry from Hong Kong to Macao, an ex-se...",A British sea captain and his crew are shipwre...
28,Bitter Sugar (Azucar Amargo),1996,Drama,0.0,A patriot (Rene Lavan) with a rebellious broth...,"A young woman, Maria, is a prostitute in the s..."
39,"Believers, The",1987,"Horror, Thriller",0.0,Mourning the accidental death of his wife and ...,A group of young people are on a camping trip ...
63,"Negotiator, The",1998,"Action, Thriller",0.0,The police try to arrest expert hostage negoti...,A former CIA agent is hired by a senator to as...
50,"Postman, The",1997,Drama,0.0,"In 2013 there are no highways, no I-ways, no d...",A man who has been living in the woods for yea...
118,Nurse Betty,2000,"Comedy, Thriller",0.0,What happens when a person decides that life i...,A nurse who has been fired from her job is hir...
113,Porky's II: The Next Day,1983,Comedy,0.0,When the students of Angel Beach High decide t...,Porky and his friends are back in this sequel ...


lowest title/year description F1


,title,year,genres,title_year_f1,reference_description,title_year_description
91,Psycho III,1986,"Horror, Thriller",0.0,Norman Bates is still running his little motel...,"After a series of murders, a young woman is ta..."
41,Raising Arizona,1987,Comedy,0.0,The Coen Brothers tell the story of a absurd y...,A man and woman who are trying to have a child...
42,Zeus and Roxanne,1997,Children's,0.0,Mary Beth is a marine biologist that gets anno...,A young boy named Jack is sent to a boarding s...
46,"Designated Mourner, The",1997,Drama,0.0,"Jack and Judy are husband and wife, and Howard...",A young man is sent to a boarding school for t...
108,Autopsy (Macchie Solari),1975,Horror,0.0,A pathology med student and a priest team up t...,A group of friends go on a trip to the country...
50,"Postman, The",1997,Drama,0.0,"In 2013 there are no highways, no I-ways, no d...",A man who has been living in the woods for yea...
26,"Rich Man's Wife, The",1996,Thriller,0.0,A rich man's wife finds she has a bad prenupti...,A young woman is hired to be the personal assi...
25,Maximum Risk,1996,"Action, Adventure, Thriller",0.0,A policeman takes his twin brother's place and...,A young man is sent to a juvenile prison for k...
107,Modern Times,1936,Comedy,0.0,The Tramp struggles to live in modern industri...,A group of unemployed workers in a factory dec...
52,"Newton Boys, The",1998,"Crime, Drama",0.0,Four Newton brothers are a poor farmer family ...,A young man is sent to a juvenile detention ce...


## 9. Save summary JSON

In [9]:
summary = {
    "run_name": RUN_NAME,
    "model_dir": str(MODEL_DIR),
    "meta_eval": meta_summary,
    "description_eval": desc_summary,
    "outputs": {
        "metadata_csv": str(OUT_DIR / "sid_to_metadata_eval.csv"),
        "description_csv": str(OUT_DIR / "description_recovery_eval.csv"),
        "summary_json": str(OUT_DIR / "summary.json"),
    },
}
with (OUT_DIR / "summary.json").open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(json.dumps(summary, indent=2, ensure_ascii=False))

{
  "run_name": "cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1",
  "model_dir": "C:\\Users\\User\\plum-ml1m-repro\\data\\processed\\artifacts\\cpt_qwen2_5_3b_base_sid_v2_plum_curriculum_v1\\final_merged",
  "meta_eval": {
    "n": 300,
    "title_accuracy": 0.7166666666666667,
    "year_accuracy": 0.7366666666666667,
    "genres_accuracy": 0.8166666666666667,
    "all_exact_accuracy": 0.71,
    "errors": 87
  },
  "description_eval": {
    "n": 120,
    "sid_only_f1_mean": 0.03153446230310105,
    "sid_only_f1_median": 0.0,
    "sid_title_f1_mean": 0.08484398788183928,
    "sid_title_f1_median": 0.059714795008912656,
    "title_year_f1_mean": 0.08792363439239673,
    "title_year_f1_median": 0.06612021857923497,
    "sid_title_beats_sid_only_rate": 0.6583333333333333,
    "title_year_beats_sid_only_rate": 0.6666666666666666
  },
  "outputs": {
    "metadata_csv": "C:\\Users\\User\\plum-ml1m-repro\\reports\\cpt_grounding\\qwen2_5_3b_sid_v2\\sid_to_metadata_eval.csv",
    "description_csv

## 10. Optional: run all items

For a full audit, restart the kernel and set:

```python
META_EVAL_N = None
DESC_EVAL_N = None
```

Then rerun from the top. The metadata test should be the primary exact grounding metric. Description scores should be interpreted together with manual samples.